# 🎬 AI Clip Maker on Google Colab
Runtime → Change runtime type → **T4 GPU** (faster Whisper, and needed for VoxCPM). Run the cells in order, then open the `trycloudflare.com` link it prints in Chrome.

Optional: add your Claude key as a Colab secret named `ANTHROPIC_API_KEY` (🔑 icon on the left) to turn on Movie Recap mode and AI highlight picking.

In [ ]:
!git clone -q https://github.com/Aung94/Hello-World.git /content/clipmaker 2>/dev/null || git -C /content/clipmaker pull -q
%cd /content/clipmaker
!apt-get -qq install -y ffmpeg fonts-noto-core > /dev/null
!pip install -q -r requirements.txt
# Optional VoxCPM voice cloning (GPU): uncomment the next line
# !pip install -q voxcpm soundfile

In [ ]:
import os, subprocess, time, re
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    print('No ANTHROPIC_API_KEY secret — clips work, Movie Recap is off')
os.environ.update(NO_BROWSER='1', HOST='0.0.0.0', PORT='8000', WHISPER_MODEL='small')
server = subprocess.Popen(['python', '-m', 'app'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        print('\n👉 Open this in Chrome:', m.group(0))
        break